In [ ]:
import ast
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
from pattern_recognition.data_preprocessing.speqle.concatenate_traces import CurveConcatenator
from pattern_recognition.data.SPEQLE_data.loadTracesSpeqle import LoadTracesSpeqle as load_traces
from pattern_recognition.data_preprocessing.speqle.concatenate_segments import SegmentsConcatenator


In [ ]:
studyType ='BL'

Curve Concatenator

In [ ]:
strain_interpDF = load_traces(studyType, 'speqle_segmentalStrainInterp_T',None)
max_length = strain_interpDF.tracesDF.groupby('study_nr').size().max()
dropna_tracesDF = strain_interpDF.tracesDF.dropna(subset=['strain_trace'])
dropna_tracesDF = dropna_tracesDF[~(dropna_tracesDF['strain_trace'].apply(lambda x: np.all(np.array(x) == 0)))]
tracesDF = dropna_tracesDF.groupby('study_nr').filter(lambda x: len(x) == max_length)

In [ ]:
concatenated_strain = CurveConcatenator(tracesDF,0)
concatenated_tracesDF = concatenated_strain.concatenate_traces()

In [ ]:
concatenated_tracesDF.head()

In [ ]:
tracesDF = concatenated_tracesDF['concatenated_trace'].tolist()

In [ ]:
array_curves = np.array(tracesDF)

In [ ]:
pca = PCA().fit(array_curves)
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.where(cumulative_explained_variance >= 0.95)[0][0] + 1
print(f'Number of components that explain 95% of variance: {n_components_95}')

In [ ]:
pca_95_var = PCA(n_components=n_components_95)
pca_result = pca_95_var.fit_transform(array_curves)

In [ ]:
#For plot PCA
explained_variance = pca_95_var.explained_variance_ratio_

fig = go.Figure(data=[
    go.Bar(y=explained_variance, x=[f'PC{i}' for i in range(1, n_components_95 + 1)])
])

fig.update_layout(
    title='Explained Variance by Principal Components',
    xaxis_title='Principal Component',
    yaxis_title='Explained Variance'
)

fig.show()

In [ ]:

# Cumulative Explained Variance Plot
cumulative_explained_variance = np.cumsum(pca_95_var.explained_variance_ratio_)

fig = go.Figure(data=[
    go.Scatter(y=cumulative_explained_variance, x=[f'PC{i}' for i in range(1, n_components_95 + 1)])
])
fig.update_layout(
    title='Cumulative Explained Variance by Principal Components',
    xaxis_title='Principal Component',
    yaxis_title='Cumulative Explained Variance'
)
fig.show()

In [ ]:
# Biplot
loadings = pca_95_var.components_.T * np.sqrt(pca_95_var.explained_variance_)

fig = go.Figure(data=[
    go.Scatter(x=pca_result[:, 1], y=pca_result[:, 2], mode='markers', name='Scores',
               marker=dict(color='blue'))
])

fig.update_layout(
    title='PCA',
    xaxis_title='PC1',
    yaxis_title='PC2',
    plot_bgcolor='white',  # Set plot background to white
    paper_bgcolor='white',  # Set paper background to white
    showlegend=True
)

fig.show()

Segment Concatenator

In [ ]:
pca_result[:, 0]


In [ ]:
concat_traces_by = 'segment'

In [ ]:
concatenated_strain = SegmentsConcatenator(seg_strain_interpDF.tracesDF)
concatenated_tracesDF = concatenated_strain.concatenate_segment_traces()

In [ ]:
unique_curve_segments = concatenated_tracesDF['curve_segment'].unique()
unique_curve_segments = unique_curve_segments[unique_curve_segments != '0']
for curve_segment in unique_curve_segments:
    segment_tracesDF = concatenated_tracesDF[
        concatenated_tracesDF['curve_segment'] == curve_segment].reset_index(drop=True)
    segment_trace = segment_tracesDF['strain_trace'].apply(ast.literal_eval)
    segment_trace = segment_trace.round(6)
    array_curves = np.array(segment_trace.tolist())   
    
    pca = PCA().fit(array_curves)
    cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
    n_components_95 = np.where(cumulative_explained_variance >= 0.95)[0][0] + 1
    print(f'Number of components that explain 95% of variance: {n_components_95}' + (
                f' in segment {curve_segment}' if concat_traces_by == 'segment' else ''))
    pca_95_var = PCA(n_components=n_components_95)
    pca_result = pca_95_var.fit_transform(array_curves)

In [ ]:
# basAnt, apAnt, midAnt, midLat, apLat, basLat, apAntSept, basAntSept, midAntSept,midInf, apInf, basInf, midPost, apPost, basPost, apSept, basSept, midSept
segment_tracesDF = concatenated_tracesDF[
        concatenated_tracesDF['curve_segment'] == 'basAnt'].reset_index(drop=True)
segment_trace = segment_tracesDF['strain_trace'].apply(ast.literal_eval)
segment_trace = segment_trace.round(6)
array_curves = np.array(segment_trace.tolist()) 

In [ ]:
pca = PCA().fit(array_curves)
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.where(cumulative_explained_variance >= 0.95)[0][0] + 1
print(f'Number of components that explain 95% of variance: {n_components_95}' + (
            f' in segment {curve_segment}' if concat_traces_by == 'segment' else ''))
pca_95_var = PCA(n_components=n_components_95)
pca_result = pca_95_var.fit_transform(array_curves)

In [ ]:
#For plot PCA
explained_variance = pca_95_var.explained_variance_ratio_

fig = go.Figure(data=[
    go.Bar(y=explained_variance, x=[f'PC{i}' for i in range(1, n_components_95 + 1)])
])

fig.update_layout(
    title='Explained Variance by Principal Components',
    xaxis_title='Principal Component',
    yaxis_title='Explained Variance'
)
fig.show()

In [ ]:

# Cumulative Explained Variance Plot
cumulative_explained_variance = np.cumsum(pca_95_var.explained_variance_ratio_)

fig = go.Figure(data=[
    go.Scatter(y=cumulative_explained_variance, x=[f'PC{i}' for i in range(1, n_components_95 + 1)])
])
fig.update_layout(
    title='Cumulative Explained Variance by Principal Components',
    xaxis_title='Principal Component',
    yaxis_title='Cumulative Explained Variance'
)
fig.show()

In [ ]:
# Biplot
loadings = pca_95_var.components_.T * np.sqrt(pca_95_var.explained_variance_)

fig = go.Figure(data=[
    go.Scatter(x=pca_result[:, 0], y=pca_result[:, 4], mode='markers', name='Scores',
               marker=dict(color='blue')),
    go.Scatter(x=loadings[:, 0], y=loadings[:, 1], mode='markers', name='Loadings',
               marker=dict(color='red'))
])

fig.update_layout(
    title='Biplot of PCA',
    xaxis_title='PC1',
    yaxis_title='PC2'
)

fig.show()

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy
from tqdm import tqdm
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import KFold